In [2]:
from dotenv import load_dotenv
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic import hub
# Import OpenAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from pprint import pprint

In [3]:
load_dotenv()  # Load environment variables from .env file
api_key = os.getenv("OPEN_AI_API_KEY")
url_to_scrap = os.getenv("URL_TO_SCRAP")
print(api_key[0:15] + "...")
print(url_to_scrap)

sk-proj-xsVKJsU...
https://books.toscrape.com/


## Scrape a web page

In [4]:
loader = WebBaseLoader(web_paths=[url_to_scrap])
docs = loader.load()

In [ ]:
for doc in docs:
    pprint(f"Document Metadata: {doc.metadata}")
    pprint(f"Document Content: {doc.page_content}")
    print("=" * 80)  # Separator between documents

## Split and do Chunking

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [7]:
splits = text_splitter.split_documents(docs)

In [8]:
# print the splits using list comprehension
pprint(len(splits))
[pprint(split) for split in splits]

12
Document(metadata={'source': 'https://books.toscrape.com/', 'title': '\n    All products | Books to Scrape - Sandbox\n', 'description': '', 'language': 'en-us'}, page_content='All products | Books to Scrape - Sandbox\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nBooks to Scrape We love being scraped!\n\n\n\n\n\n\n\n\nHome\n\nAll products\n\n\n\n\n\n\n\n\n\n                            \n                                Books\n                            \n                        \n\n\n\n                            \n                                Travel\n                            \n                        \n\n\n\n                            \n                                Mystery\n                            \n                        \n\n\n\n                            \n                                Historical Fiction\n                            \n                        \n\n\n\n                            \n                                Sequential Art\n                            \n 

[None, None, None, None, None, None, None, None, None, None, None, None]

## Save the chunks to vectorstore using OpenAIEmbeddings

An API key is required

In [ ]:
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=OpenAIEmbeddings(api_key=api_key)
)

In [ ]:
print(f"Vectorstore created with {vectorstore.collection.count()} documents.")

In [ ]:
ids = vectorstore._collection.get()
print(ids)

In [ ]:
for i, id in enumerate(ids):
    print(f"Document # {i+1:02d} | ID: {id} | Content: {vectorstore._collection.get(id)}")

## Retrieve using Context Search

In [41]:
# Retrieve an existing prompt from LangSmith
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "rlm/rag-prompt", 
    dangerously_pull_public_prompt=True
)

type(prompt)

langchain_core.prompts.chat.ChatPromptTemplate

In [48]:
pprint(prompt.messages[0].prompt.template)

('You are an assistant for question-answering tasks. Use the following pieces '
 "of retrieved context to answer the question. If you don't know the answer, "
 "just say that you don't know. Use three sentences maximum and keep the "
 'answer concise.\n'
 'Question: {question} \n'
 'Context: {context} \n'
 'Answer:')
